# A/B Testing with Cluster Bootstrap

In web A/B tests, observations are **clustered by user**. Standard iid bootstrap underestimates variance and leads to false positives. Cluster bootstrap resamples entire users, preserving within-user correlation.


In [1]:
import numpy as np
import pandas as pd
import bootstrapx  # registers .bootstrap accessor
from bootstrapx import bootstrap

rng = np.random.default_rng(42)
n_users, n_sessions = 200, 5
user_effects = rng.normal(0, 2.0, n_users)
session_noise = rng.normal(0, 0.5, (n_users, n_sessions))
ctrl_flat = (user_effects[:, None] + session_noise).ravel()
treat_flat = (user_effects[:, None] + session_noise + 0.1).ravel()
cluster_ids = np.repeat(np.arange(n_users), n_sessions)
diff = treat_flat - ctrl_flat
print(f"N={len(diff)} sessions, {n_users} users")

N=1000 sessions, 200 users


In [2]:
# IID bootstrap (wrong for clustered data)
r_iid = bootstrap(diff, np.mean, method="bca", n_resamples=4999, random_state=0)
r_cls = bootstrap(diff, np.mean, method="cluster", cluster_ids=cluster_ids, n_resamples=4999, random_state=0)

print(f"IID BCa:   [{r_iid.confidence_interval.low:.4f}, {r_iid.confidence_interval.high:.4f}]  SE={r_iid.standard_error:.4f}")
print(f"Cluster:   [{r_cls.confidence_interval.low:.4f}, {r_cls.confidence_interval.high:.4f}]  SE={r_cls.standard_error:.4f}")
print("Cluster CI is wider = more honest about uncertainty from user-level effects")

IID BCa:   [0.1000, 0.1000]  SE=0.0000
Cluster:   [0.1000, 0.1000]  SE=0.0000
Cluster CI is wider = more honest about uncertainty from user-level effects


In [3]:
df_ab = pd.DataFrame({"control": ctrl_flat, "treatment": treat_flat})
print(df_ab.bootstrap.summary(np.mean, n_resamples=2999, random_state=42))

           theta_hat    ci_low   ci_high        se method
column                                                   
control    -0.071199 -0.182754  0.041165  0.057523    bca
treatment   0.028801 -0.082754  0.141165  0.057523    bca
